# EMTscore Analysis and Plots

## 2. Example Walkthrough (Bulk RNA-seq)

### 2.1 Load Cell/Sample Annotation

In [ ]:
import importlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

cell_annot = pd.read_csv("data/cell_annotation_file.csv")
cell_annot.head()

### 2.2 Load Gene Expression Data

In [ ]:
# Rows = samples, columns = genes (transposed for scoring)
geneExp = pd.read_csv("data/geneExp.csv", index_col=0).T
print(f"geneExp shape: {geneExp.shape}  (samples × genes)")
geneExp.iloc[:4, :5]

### 2.3 Load Gene Signature Data

In [ ]:
E_sig = pd.read_csv("data/Panchy_et_al_E_signature.csv")
M_sig = pd.read_csv("data/Panchy_et_al_M_signature.csv")
print("E signature:", E_sig.shape)
print(E_sig.head())
print("\nM signature:", M_sig.shape)
print(M_sig.head())

### 2.4 Prepare Gene Sets and Save as GMT

In [ ]:
E_genes = E_sig["GeneName"].tolist()
M_genes = M_sig["GeneName"].tolist()

gmt_path = "data/EM_signature.gmt"
with open(gmt_path, "w") as f:
    f.write("Panchy_et_al_E_signature\tNA\t" + "\t".join(E_genes) + "\n")
    f.write("Panchy_et_al_M_signature\tNA\t" + "\t".join(M_genes) + "\n")

print(f"GMT saved: {gmt_path}")

### 2.5 Compute Scores Using Multiple Methods

In [ ]:
import importlib
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import emtscore.nnpca, emtscore.nsprcomp, emtscore.scse
importlib.reload(emtscore.nnpca)
importlib.reload(emtscore.nsprcomp)
importlib.reload(emtscore.scse)

from emtscore.nnpca   import run_nnPCA, execute_nnPCA_single
from emtscore.ssGSEA  import execute_ssgsva, execute_ssgsea_single
from emtscore.aucell  import execute_aucell, execute_aucell_single
from emtscore.jasmine import execute_jasmine_single, jasmine, parse_gmt as _jas_parse_gmt
from emtscore.scse    import execute_scse
from emtscore.nsprcomp import compute_M1_M2_scores


# ============================================================
# Step 1 - Run all five methods 
# ============================================================
GMT = "data/EM_signature.gmt"

nnPCA_scores  = run_nnPCA(geneExp,  gmt_file=GMT, dimension=1)
aucell_scores = execute_aucell(geneExp, gmt_file=GMT)
ssgsea_scores = execute_ssgsva(geneExp, gmt_file=GMT)
scse_scores   = execute_scse(geneExp,   gmt_file=GMT)


def _jasmine_per_gene_set(expr, gmt_file):
    """Run JASMINE separately on each gene set in a GMT -> (samples x sets) df."""
    # parse_gmt above returns a flattened union; we need per-set results.
    sets = {}
    with open(gmt_file) as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 3:
                sets[parts[0]] = [g for g in parts[2:] if g]

    # jasmine() takes (genes x samples); expr is (samples x genes) -> transpose
    expr_T = expr.T
    out = {}
    for name, genes in sets.items():
        jas = jasmine(expr_T, genes)
        if jas.empty:
            out[name] = pd.Series(0.0, index=expr.index)
        else:
            out[name] = jas["JASMINE"].reindex(expr.index).fillna(0.0)
    return pd.DataFrame(out, index=expr.index)


jasmine_scores = _jasmine_per_gene_set(geneExp, GMT)

print("nnPCA  :", nnPCA_scores.shape,  list(nnPCA_scores.columns))
print("AUCell :", aucell_scores.shape, list(aucell_scores.columns))
print("ssGSEA :", ssgsea_scores.shape, list(ssgsea_scores.columns))
print("JASMINE:", jasmine_scores.shape, list(jasmine_scores.columns))
print("SCSE   :", scse_scores.shape,    list(scse_scores.columns))


# ============================================================
# Step 2 - Standardize each method's output
# ============================================================
def zscore(df):
    df = df.astype(float)
    keep = df.std(axis=0, skipna=True).fillna(0) > 0
    z = pd.DataFrame(0.0, index=df.index, columns=df.columns)
    if keep.any():
        sub = df.loc[:, keep].fillna(df.loc[:, keep].mean())
        z.loc[:, keep] = StandardScaler().fit_transform(sub)
    return z


nnPCA_z   = zscore(nnPCA_scores)
aucell_z  = zscore(aucell_scores)
ssgsea_z  = zscore(ssgsea_scores)
jasmine_z = zscore(jasmine_scores)
scse_z    = zscore(scse_scores)


# ============================================================
# Step 3 - Collapse multi-score methods 
# ============================================================
def collapse(df):
    return df.mean(axis=1)


# ============================================================
# Step 4 - Final five-method comparison table
# ============================================================
comparison = pd.DataFrame({
    "nnPCA":   collapse(nnPCA_z),
    "AUCell":  collapse(aucell_z),
    "ssGSEA":  collapse(ssgsea_z),
    "JASMINE": collapse(jasmine_z),
    "SCSE":    collapse(scse_z),
})

print(comparison.head())


### 2.6 Prepare Data for Plotting

In [ ]:
# ============================================================
# SAFE PLOT PREPARATION 
# ============================================================

def prepare_plot_data(scores_df, annotations_df):
    annotations_df = annotations_df.copy()

    if "name" in annotations_df.columns:
        annotations_df = annotations_df.set_index("name")

    common = scores_df.index.intersection(annotations_df.index)

    scores_df = scores_df.loc[common]
    annotations_df = annotations_df.loc[common]

    return annotations_df.join(scores_df)


# --- rename scores for the 5 methods --------------------------------
nnPCA_scores = nnPCA_scores.rename(columns={
    "Escore": "Panchy_et_al_E_signature",
    "Mscore": "Panchy_et_al_M_signature",
})

aucell_scores.columns  = ["Panchy_et_al_E_signature", "Panchy_et_al_M_signature"]
ssgsea_scores.columns  = ["Panchy_et_al_E_signature", "Panchy_et_al_M_signature"]
jasmine_scores.columns = ["Panchy_et_al_E_signature", "Panchy_et_al_M_signature"]
scse_scores.columns    = ["Panchy_et_al_E_signature", "Panchy_et_al_M_signature"]

# --- M1/M2 ---
M_genes_filtered = [g for g in M_sig["GeneName"].tolist() if g in geneExp.columns]
M_scores_df = compute_M1_M2_scores(geneExp, M_genes_filtered, perturbation=1e-4)

# --- BUILD FINAL DATASETS ---
nnPCA_em   = prepare_plot_data(nnPCA_scores,   cell_annot)
aucell_em  = prepare_plot_data(aucell_scores,  cell_annot)
ssgsea_em  = prepare_plot_data(ssgsea_scores,  cell_annot)
jasmine_em = prepare_plot_data(jasmine_scores, cell_annot)
scse_em    = prepare_plot_data(scse_scores,    cell_annot)
nnPCA_mm   = prepare_plot_data(M_scores_df,    cell_annot)

# --- COLORS ---
COLORS = ["#F87189","#CE9031","#A48CF5","#97A430","#39A7D0","#E57D5F",
          "#84C7B9","#E1AF64","#C26CCF","#B0BF43","#57C3E8","#F29D9E","#92AAE6"]

# --- DEBUG PRINTS (NOW SAFE) ---
print("All data prepared. Shapes:")
print(f"  nnPCA_em:   {nnPCA_em.shape}")
print(f"  aucell_em:  {aucell_em.shape}")
print(f"  ssgsea_em:  {ssgsea_em.shape}")
print(f"  jasmine_em: {jasmine_em.shape}")
print(f"  scse_em:    {scse_em.shape}")
print(f"  nnPCA_mm:   {nnPCA_mm.shape}")


### 2.7 E Score vs M Score Plot

Scatter plot with KDE contours and mean ± SD markers per cell type.

In [ ]:
print(nnPCA_em.head())

In [ ]:
print(cell_annot.columns)
print(cell_annot.index[:5])

In [ ]:
if "name" in cell_annot.columns:
    cell_annot = cell_annot.set_index("name")

common = geneExp.index.intersection(cell_annot.index)
geneExp    = geneExp.loc[common]
cell_annot = cell_annot.loc[common]


from emtscore.nnpca import parse_gmt, get_nnPCA_result

_genesets = parse_gmt("data/EM_signature.gmt")
nnPCA_scores = pd.DataFrame(
    {name: get_nnPCA_result(geneExp, genes, align_direction="positive")
     for name, genes in _genesets.items()},
    index=geneExp.index,
)

aucell_scores = execute_aucell(geneExp, gmt_file="data/EM_signature.gmt")
ssgsea_scores = execute_ssgsva(geneExp, gmt_file="data/EM_signature.gmt")

nnPCA_em  = nnPCA_scores.join(cell_annot["celltype_annotation"])
aucell_em = aucell_scores.join(cell_annot["celltype_annotation"])
ssgsea_em = ssgsea_scores.join(cell_annot["celltype_annotation"])

print("nnPCA_em columns:", list(nnPCA_em.columns))
print("aucell_em columns:", list(aucell_em.columns))
print("ssgsea_em columns:", list(ssgsea_em.columns))


In [ ]:
print("\n=== nnPCA_em DEBUG TRACE ===")
print("TYPE:", type(nnPCA_em))
print("SHAPE:", getattr(nnPCA_em, "shape", None))
print("COLUMNS:", getattr(nnPCA_em, "columns", None))
print("HEAD:\n", nnPCA_em.head() if hasattr(nnPCA_em, "head") else nnPCA_em)

print("\n=== COLUMN CHECK ===")
print("Has Escore:", "Escore" in getattr(nnPCA_em, "columns", []))
print("Has Mscore:", "Mscore" in getattr(nnPCA_em, "columns", []))
print("Has Panchy_E:", "Panchy_et_al_E_signature" in getattr(nnPCA_em, "columns", []))

In [ ]:
# ============================================================
# E vs M scatter (nnPCA / AUCell / ssGSEA)
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Palette from the reference EMTscore.html
REF_PALETTE = [
    "#F87189", "#CE9031", "#A48CF5", "#97A430", "#39A7D0",
    "#E57D5F", "#84C7B9", "#E1AF64", "#C26CCF", "#B0BF43",
    "#57C3E8", "#F29D9E", "#92AAE6",
]


def plot_em(data, xcol, ycol, title, ax=None, palette=None):
    """E-vs-M scatter with KDE contours and mean +/- SD crosses."""
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(8, 6))

    # Safety check
    if xcol not in data.columns or ycol not in data.columns:
        print(f"[SKIP] {title}")
        print("Missing columns:", xcol, ycol)
        print("Available:", list(data.columns))
        return

    cell_types = sorted(data["celltype_annotation"].dropna().unique())
    colors = (palette or REF_PALETTE)[: len(cell_types)]
    color_map = dict(zip(cell_types, colors))

    # KDE contour fills per cell type
    for ct in cell_types:
        subset = data[data["celltype_annotation"] == ct]
        if len(subset) > 5:
            sns.kdeplot(
                data=subset, x=xcol, y=ycol,
                fill=True, alpha=0.18, levels=5, thresh=0.05,
                color=color_map[ct], ax=ax,
            )

    # Scatter points
    sns.scatterplot(
        data=data, x=xcol, y=ycol,
        hue="celltype_annotation", hue_order=cell_types,
        palette=color_map, s=36, alpha=0.55, edgecolor=None, ax=ax,
    )

    # Mean +/- SD markers (black cross behind, colored disc with white border on top)
    for ct in cell_types:
        subset = data[data["celltype_annotation"] == ct]
        mX, mY = subset[xcol].mean(), subset[ycol].mean()
        sX, sY = subset[xcol].std(),  subset[ycol].std()

        ax.errorbar(
            mX, mY, xerr=sX, yerr=sY,
            fmt="none", ecolor="black",
            elinewidth=3, capsize=0, zorder=5,
        )
        ax.scatter(
            mX, mY, s=220, color=color_map[ct],
            edgecolor="white", linewidth=2.2, zorder=6,
        )

    # Reference-like theme: white bg, black border, no grid, italic labels
    ax.set_xlabel(xcol, fontsize=13, style="italic")
    ax.set_ylabel(ycol, fontsize=13, style="italic")
    ax.set_title(title, fontsize=14)
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_color("black")
        spine.set_linewidth(1.2)
    ax.set_facecolor("white")

    # Legend with italic title, outside the axes
    leg = ax.legend(
        title="celltype_annotation",
        bbox_to_anchor=(1.02, 1), loc="upper left",
        frameon=True, fontsize=10, title_fontsize=11,
    )
    if leg is not None:
        leg.get_title().set_fontstyle("italic")

    if standalone:
        plt.tight_layout()
        plt.show()


# --- Calls ---
E_COL = "Panchy_et_al_E_signature"
M_COL = "Panchy_et_al_M_signature"

plot_em(nnPCA_em,  E_COL, M_COL, "E vs M Scores (nnPCA)")
plot_em(aucell_em, E_COL, M_COL, "E vs M Scores (AUCell)")
plot_em(ssgsea_em, E_COL, M_COL, "E vs M Scores (ssGSEA)")


### 2.8 M1 vs M2 Score Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
cell_types = nnPCA_mm["celltype_annotation"].unique()

for i, ct in enumerate(cell_types):
    subset = nnPCA_mm[nnPCA_mm["celltype_annotation"] == ct]
    if len(subset) > 5:
        sns.kdeplot(data=subset, x="M1", y="M2",
                    fill=True, alpha=0.25, levels=6, thresh=0.05,
                    color=COLORS[i % len(COLORS)], zorder=1, ax=ax)

sns.scatterplot(data=nnPCA_mm, x="M1", y="M2", hue="celltype_annotation",
                palette={ct: COLORS[i % len(COLORS)] for i, ct in enumerate(cell_types)},
                s=35, alpha=0.6, edgecolor=None, zorder=2, ax=ax)

for i, ct in enumerate(cell_types):
    subset = nnPCA_mm[nnPCA_mm["celltype_annotation"] == ct]
    m1, m2 = subset["M1"].mean(), subset["M2"].mean()
    s1, s2 = subset["M1"].std(),  subset["M2"].std()
    ax.errorbar(m1, m2, xerr=s1, yerr=s2,
                fmt="none", ecolor="black", elinewidth=3, capsize=0, zorder=3)
    ax.scatter(m1, m2, s=260, color=COLORS[i % len(COLORS)],
               edgecolor="white", linewidth=2, zorder=4)

ax.set_xlabel("M1 Score", fontsize=12)
ax.set_ylabel("M2 Score", fontsize=12)
ax.set_title("M1 vs M2 Scores (nnPCA on M signature)", fontsize=14)
ax.legend(title="Cell Type", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 2.9 Combined Plot (E vs M  |  M1 vs M2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_em(nnPCA_em, E_COL, M_COL, "E vs M Scores (nnPCA)", ax=axes[0])

ax = axes[1]
cell_types = nnPCA_mm["celltype_annotation"].unique()
for i, ct in enumerate(cell_types):
    subset = nnPCA_mm[nnPCA_mm["celltype_annotation"] == ct]
    if len(subset) > 5:
        sns.kdeplot(data=subset, x="M1", y="M2",
                    fill=True, alpha=0.25, levels=6, thresh=0.05,
                    color=COLORS[i % len(COLORS)], ax=ax)
sns.scatterplot(data=nnPCA_mm, x="M1", y="M2", hue="celltype_annotation",
                palette={ct: COLORS[i % len(COLORS)] for i, ct in enumerate(cell_types)},
                s=35, alpha=0.6, edgecolor=None, ax=ax)
for i, ct in enumerate(cell_types):
    subset = nnPCA_mm[nnPCA_mm["celltype_annotation"] == ct]
    m1, m2 = subset["M1"].mean(), subset["M2"].mean()
    s1, s2 = subset["M1"].std(),  subset["M2"].std()
    ax.errorbar(m1, m2, xerr=s1, yerr=s2, fmt="none", ecolor="black", elinewidth=3, capsize=0, zorder=3)
    ax.scatter(m1, m2, s=260, color=COLORS[i % len(COLORS)], edgecolor="white", linewidth=2, zorder=4)
ax.set_xlabel("M1 Score", fontsize=12)
ax.set_ylabel("M2 Score", fontsize=12)
ax.set_title("M1 vs M2 Scores (nnPCA on M signature)", fontsize=14)
ax.legend(title="Cell Type", bbox_to_anchor=(1.05, 1), loc="upper left")

fig.suptitle("Panchy_et_al", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

### 2.10 E/M Score Distributions (M1 Histogram)

In [ ]:
nnPCA_mm["celltype_annotation"] = nnPCA_mm["celltype_annotation"].astype(str)
cell_types = nnPCA_mm["celltype_annotation"].unique()
palette = {ct: COLORS[i % len(COLORS)] for i, ct in enumerate(cell_types)}

plt.figure(figsize=(10, 6))
sns.histplot(data=nnPCA_mm, x="M1", hue="celltype_annotation",
             multiple="layer", palette=palette, alpha=0.6, bins=30, edgecolor=None)
plt.xlabel("M1 Score")
plt.ylabel("Count")
plt.title("Distribution of M1 Scores Across Cell Types")
plt.legend(title="Cell Type", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 2.11 Heatmap (M Signature Genes, Hierarchically Clustered)

#### 2.11.0 Full M-Signature Heatmap (All Genes)

Mirrors `plot_heatmap_function(t(geneExp), Panchy_et_al_M_signature)` in
the R vignette: every gene in the Panchy M signature that is detected in
`geneExp`, hierarchically clustered across samples.  The next cell (the
top-10 PC1 / PC2 driver-gene heatmap) zooms into the main loading genes.


In [ ]:
# ------------------------------------------------------------------
# 2.11.0 - Full M-signature heatmap (all Panchy M genes detected)
#   Hierarchical clustering on samples; sample columns colored by
#   celltype_annotation.
# ------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.cluster.hierarchy import linkage, leaves_list, dendrogram
from scipy.spatial.distance import pdist

M_full = [g for g in M_sig["GeneName"].tolist() if g in geneExp.columns]
print(f"Full-M heatmap: {len(M_full)} of {len(M_sig)} signature genes present")

# genes x samples
expr_full = geneExp[M_full].T

# Restrict to samples we have annotation for
valid_samples = [s for s in expr_full.columns if s in nnPCA_em.index.values]
expr_full = expr_full[valid_samples]

# Cluster both rows (genes) and columns (samples)
col_order = leaves_list(linkage(pdist(expr_full.T.values), method="average"))
row_order = leaves_list(linkage(pdist(expr_full.values),   method="average"))
expr_ord  = expr_full.iloc[row_order, col_order]
sample_order = expr_ord.columns.tolist()
gene_order   = expr_ord.index.tolist()

# --- figure layout: dendrogram | annotation strip | heatmap ---------
n_genes   = len(gene_order)
n_samples = len(sample_order)
fig_w = max(14, n_samples * 0.09)
fig_h = max(10, n_genes   * 0.10 + 3)
fig = plt.figure(figsize=(fig_w, fig_h))

left, hm_w = 0.08, 0.82
top = 0.97
dend_h = 0.10; dend_b = top - dend_h
anno_h = 0.018; anno_b = dend_b - anno_h - 0.004
hm_h   = 0.80; hm_b   = anno_b - hm_h - 0.004

ax_dend = fig.add_axes([left, dend_b, hm_w, dend_h])
ax_anno = fig.add_axes([left, anno_b, hm_w, anno_h])
ax_hm   = fig.add_axes([left, hm_b,   hm_w, hm_h])

col_link = linkage(pdist(expr_full.T.values[col_order]), method="average")
dendrogram(col_link, ax=ax_dend, color_threshold=0,
           above_threshold_color="black", no_labels=True)
ax_dend.set_facecolor("white"); ax_dend.axis("off")

# Sample-level celltype strip
cell_types = nnPCA_em.loc[sample_order, "celltype_annotation"].values
unique_ct  = sorted(pd.Series(cell_types).dropna().unique())
ct_palette = dict(zip(unique_ct, COLORS[: len(unique_ct)]))
anno_rgb = np.array([[mcolors.to_rgb(ct_palette.get(c, "#CCCCCC"))]
                      for c in cell_types]).transpose(1, 0, 2)
ax_anno.imshow(anno_rgb, aspect="auto", interpolation="none")
ax_anno.set_xticks([]); ax_anno.set_yticks([])
for sp in ax_anno.spines.values(): sp.set_visible(False)

# Main heatmap: z-scored per gene
Z = expr_ord.subtract(expr_ord.mean(axis=1), axis=0).divide(
        expr_ord.std(axis=1).replace(0, 1), axis=0)

vmax = float(np.nanpercentile(np.abs(Z.values), 99)) or 1.0
norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)
hm_img = ax_hm.imshow(Z.values, aspect="auto", cmap="RdBu_r",
                      norm=norm, interpolation="none")
ax_hm.set_xticks([]); ax_hm.set_yticks([])
for sp in ax_hm.spines.values(): sp.set_visible(False)

# Gene labels on the right
ax_hm_r = ax_hm.twinx()
ax_hm_r.set_ylim(ax_hm.get_ylim())
# Only show a subset of labels if there are many genes
label_stride = max(1, n_genes // 60)
tick_pos = list(range(0, n_genes, label_stride))
ax_hm_r.set_yticks(tick_pos)
ax_hm_r.set_yticklabels([gene_order[i] for i in tick_pos], fontsize=6)
ax_hm_r.tick_params(right=False, length=0)
for sp in ax_hm_r.spines.values(): sp.set_visible(False)

# Colorbar + cell-type legend on the right
cb_ax = fig.add_axes([left + hm_w + 0.015, hm_b + 0.3, 0.015, 0.2])
cb = plt.colorbar(hm_img, cax=cb_ax)
cb.set_label("Expr (z-score)", fontsize=8)
cb.ax.tick_params(labelsize=7)

leg_ax = fig.add_axes([left + hm_w + 0.015, hm_b + 0.55, 0.12, 0.20])
leg_ax.axis("off")
handles = [plt.Line2D([], [], marker="s", linestyle="None",
                      color=ct_palette[c], markersize=8, label=c)
           for c in unique_ct]
leg_ax.legend(handles=handles, title="Cell type",
              fontsize=7, title_fontsize=8, loc="upper left",
              frameon=False)

fig.suptitle("Panchy et al. M-signature - full heatmap",
             fontsize=13, fontweight="bold", y=0.995)
plt.show()


In [ ]:
from scipy.cluster.hierarchy import linkage, leaves_list, dendrogram
from scipy.spatial.distance import pdist
from numpy.linalg import svd as npsvd
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.colorbar import ColorbarBase

top_pc1 = [g for g in ["VIM","LGALS1","FSTL1","MSN","CAV1","TPM2","CALD1","PDGFC","WWTR1","EMP3"]
           if g in geneExp.columns]
top_pc2 = [g for g in ["MFAP4","CXCR4","FXYD6","SPARC","TCF4","IGFBP5","TUBA1A","FHL1","FYN","DPYSL3"]
           if g in geneExp.columns]
all_top_genes = top_pc1 + top_pc2
gene_pc_label = {g: "PC1" for g in top_pc1}
gene_pc_label.update({g: "PC2" for g in top_pc2})

# Loadings via SVD
M_genes_present = [g for g in M_sig["GeneName"].tolist() if g in geneExp.columns]
X_c = geneExp[M_genes_present].values - geneExp[M_genes_present].values.mean(axis=0)
U, s_sv, Vt = npsvd(X_c, full_matrices=False)
pc1_loadings = pd.Series(np.abs(Vt[0]), index=M_genes_present)
pc2_loadings = pd.Series(np.abs(Vt[1]), index=M_genes_present)
pc1_bar = pc1_loadings[all_top_genes].values
pc2_bar = pc2_loadings[all_top_genes].values

# Submatrix & clustering
expr_sub = geneExp[all_top_genes].T
valid_in_expr = [s for s in geneExp.index if s in nnPCA_mm.index.values]
expr_filtered = expr_sub[valid_in_expr]
col_order = leaves_list(linkage(pdist(expr_filtered.T.values), method="average"))
expr_ordered = expr_filtered.iloc[:, col_order]
sample_order = expr_ordered.columns.tolist()

valid_samples  = [s for s in sample_order if s in nnPCA_mm.index.values]
scores_aligned = nnPCA_mm.loc[valid_samples]
m1_vals = scores_aligned["M1"].values
m2_vals = scores_aligned["M2"].values
expr_ordered = expr_ordered[valid_samples]

# Norms
expr_vals = expr_ordered.values
data_min, data_max = expr_vals.min(), expr_vals.max()
data_mid = (data_min + data_max) / 2
norm_hm  = mcolors.TwoSlopeNorm(vmin=data_min, vcenter=data_mid, vmax=data_max)

n_genes, n_samples = len(all_top_genes), len(valid_samples)
fig_w = max(12, n_samples * 0.13)
fig_h = max(8,  n_genes   * 0.38 + 3)
fig   = plt.figure(figsize=(fig_w, fig_h))

left_hm = 0.09; hm_w = 0.67
pc1_l = left_hm + hm_w + 0.008; pc_w = 0.022
pc2_l = pc1_l + pc_w + 0.005
cb_l  = pc2_l + pc_w + 0.045
top   = 0.96
dend_h = 0.12; dend_b = top - dend_h
m1_h = 0.034;  m1_b   = dend_b - m1_h - 0.004
m2_h = 0.034;  m2_b   = m1_b   - m2_h - 0.003
hm_h = 0.72;   hm_b   = m2_b   - hm_h - 0.004

col_link = linkage(pdist(expr_filtered.T.values[col_order]), method="average")

ax_dend  = fig.add_axes([left_hm,   dend_b, hm_w,   dend_h])
ax_m1    = fig.add_axes([left_hm,   m1_b,   hm_w,   m1_h])
ax_m2    = fig.add_axes([left_hm,   m2_b,   hm_w,   m2_h])
ax_label = fig.add_axes([0.04,      hm_b,   0.035,  hm_h])
ax_hm    = fig.add_axes([left_hm,   hm_b,   hm_w,   hm_h])
ax_pc1   = fig.add_axes([pc1_l,     hm_b,   pc_w,   hm_h])
ax_pc2   = fig.add_axes([pc2_l,     hm_b,   pc_w,   hm_h])

dendrogram(col_link, ax=ax_dend, color_threshold=0,
           above_threshold_color="black", no_labels=True)
ax_dend.set_facecolor("white"); ax_dend.axis("off")

m1_norm = (m1_vals - m1_vals.min()) / (m1_vals.max() - m1_vals.min() + 1e-9)
ax_m1.imshow(np.array([[plt.cm.Greys(v) for v in m1_norm]]), aspect="auto", interpolation="none")
ax_m1.set_yticks([0]); ax_m1.set_yticklabels(["M_PC1_score"], fontsize=7)
ax_m1.set_xticks([]); ax_m1.tick_params(left=False, length=0)
for sp in ax_m1.spines.values(): sp.set_visible(False)

m2_norm = (m2_vals - m2_vals.min()) / (m2_vals.max() - m2_vals.min() + 1e-9)
ax_m2.imshow(np.array([[plt.cm.RdPu(0.1 + 0.9 * v) for v in m2_norm]]), aspect="auto", interpolation="none")
ax_m2.set_yticks([0]); ax_m2.set_yticklabels(["M_PC2_score"], fontsize=7)
ax_m2.set_xticks([]); ax_m2.tick_params(left=False, length=0)
for sp in ax_m2.spines.values(): sp.set_visible(False)

label_colors = ["#2166AC" if gene_pc_label.get(g) == "PC1" else "#D6604D" for g in all_top_genes]
ax_label.imshow(np.array([[mcolors.to_rgb(c)] for c in label_colors]), aspect="auto", interpolation="none")
ax_label.set_xticks([]); ax_label.set_yticks([])
ax_label.set_xlabel("Label", fontsize=7, labelpad=3)
for sp in ax_label.spines.values(): sp.set_visible(False)

hm_img = ax_hm.imshow(expr_ordered.values, aspect="auto", cmap="RdBu_r", norm=norm_hm, interpolation="none")
ax_hm.set_xticks([]); ax_hm.set_yticks([])
for sp in ax_hm.spines.values(): sp.set_visible(False)
ax_hm_r = ax_hm.twinx()
ax_hm_r.set_ylim(ax_hm.get_ylim())
ax_hm_r.set_yticks(range(n_genes)); ax_hm_r.set_yticklabels(all_top_genes, fontsize=8)
ax_hm_r.tick_params(right=False, length=0)
for sp in ax_hm_r.spines.values(): sp.set_visible(False)

pc1_norm = pc1_bar / (pc1_bar.max() + 1e-9)
ax_pc1.imshow(np.array([[plt.cm.Blues(v)] for v in pc1_norm]), aspect="auto", interpolation="none")
ax_pc1.set_xticks([]); ax_pc1.set_yticks([]); ax_pc1.set_xlabel("PC1", fontsize=7, labelpad=3)
for sp in ax_pc1.spines.values(): sp.set_visible(False)

pc2_norm = pc2_bar / (pc2_bar.max() + 1e-9)
ax_pc2.imshow(np.array([[plt.cm.Reds(v)] for v in pc2_norm]), aspect="auto", interpolation="none")
ax_pc2.set_xticks([]); ax_pc2.set_yticks([]); ax_pc2.set_xlabel("PC2", fontsize=7, labelpad=3)
for sp in ax_pc2.spines.values(): sp.set_visible(False)

# Colorbars
ax_leg = fig.add_axes([cb_l, top - 0.09, 0.12, 0.08])
ax_leg.axis("off")
ax_leg.legend(handles=[mpatches.Patch(color="#2166AC", label="M_PC1"),
                        mpatches.Patch(color="#D6604D", label="M_PC2")],
              title="Label", fontsize=7, title_fontsize=8, loc="upper left", frameon=False)
ax_cb1 = fig.add_axes([cb_l, top-0.30, 0.025, 0.17])
ColorbarBase(ax_cb1, cmap=plt.cm.Greys,
             norm=mcolors.Normalize(vmin=m1_vals.min(), vmax=m1_vals.max()),
             orientation="vertical").set_label("M_PC1_score", fontsize=7)
ax_cb1.tick_params(labelsize=6)
ax_cb2 = fig.add_axes([cb_l, top-0.52, 0.025, 0.17])
cb2 = ColorbarBase(ax_cb2, cmap=plt.cm.RdBu_r,
                   norm=mcolors.TwoSlopeNorm(vmin=data_min, vcenter=data_mid, vmax=data_max),
                   orientation="vertical")
cb2.set_label("Gene Expr", fontsize=7)
cb2.set_ticks([data_min, data_mid, data_max])
cb2.set_ticklabels([f"{data_min:.0f}", f"{data_mid:.0f}", f"{data_max:.0f}"])
ax_cb2.tick_params(labelsize=6)
ax_cb3 = fig.add_axes([cb_l, top-0.69, 0.025, 0.13])
ColorbarBase(ax_cb3, cmap=plt.cm.Blues,
             norm=mcolors.Normalize(vmin=0, vmax=pc1_bar.max()),
             orientation="vertical").set_label("PC1", fontsize=7)
ax_cb3.tick_params(labelsize=6)
ax_cb4 = fig.add_axes([cb_l, top-0.84, 0.025, 0.11])
ColorbarBase(ax_cb4, cmap=plt.cm.RdPu,
             norm=mcolors.Normalize(vmin=m2_vals.min(), vmax=m2_vals.max()),
             orientation="vertical").set_label("M_PC2_score", fontsize=7)
ax_cb4.tick_params(labelsize=6)
plt.show()

---
## 3. Example Walkthrough: Single-Cell RNA-seq

### 3.1 Data Importing and Basic Calculations

In [ ]:
from utility.load_cook2020 import load_all_cook2020

adatas = load_all_cook2020()
for name, adata in adatas.items():
    print(f"{name}: {adata.shape},  Pseudotime: "
          f"{adata.obs['Pseudotime'].min():.3f} – {adata.obs['Pseudotime'].max():.3f}")

### 3.2 Building Gaussian Mixture Models (GMM) in E-M Space

For each dataset: log-normalise → scale → nnPCA → align to pseudotime → GMM (3 states: E, EM1, M) → scatter + Sankey.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture

from emtscore.nsprcomp import nsprcomp


def lognorm(expr, scale_factor=10000):
    lib_size = expr.sum(axis=1).values[:, None]
    lib_size = np.where(lib_size == 0, 1, lib_size)
    return np.log1p(expr.values / lib_size * scale_factor)


def scale_genes(X):
    mean = X.mean(axis=0)
    std = X.std(axis=0)
    std[std == 0] = 1
    return (X - mean) / std


def build_gmm_em(df, E_genes, M_genes):
    """Return (em_matrix, state_labels) for one dataset."""
    meta_cols = ["Pseudotime", "Treatment", "Time", "Cluster"]
    expr_cols = df.columns.difference(meta_cols)
    pt = df["Pseudotime"].values

    full_expr = lognorm(df[expr_cols])
    full_expr = pd.DataFrame(full_expr, index=df.index, columns=expr_cols)

    E_mat = scale_genes(full_expr[[g for g in E_genes if g in full_expr.columns]].values)
    M_mat = scale_genes(full_expr[[g for g in M_genes if g in full_expr.columns]].values)

    E_raw = nsprcomp(E_mat, ncomp=1)["x"][:, 0]
    M_raw = nsprcomp(M_mat, ncomp=1)["x"][:, 0]

    if np.corrcoef(pt, E_raw)[0, 1] > 0:
        E_raw = -E_raw
    if np.corrcoef(pt, M_raw)[0, 1] < 0:
        M_raw = -M_raw

    E_score = (E_raw - E_raw.mean()) / E_raw.std()
    M_score = (M_raw - M_raw.mean()) / M_raw.std()
    em = np.column_stack([E_score, M_score])

    gmm = GaussianMixture(n_components=3, random_state=42, n_init=5)
    gmm.fit(em)
    labels = gmm.predict(em)
    order = np.argsort(gmm.means_[:, 0])
    state_map = {order[0]: "M", order[1]: "EM1", order[2]: "E"}
    state_labels = np.array([state_map[l] for l in labels])
    return em, state_labels


# --- Gene signatures  -----------------------
E_genes = pd.read_csv("data/Panchy_et_al_E_signature.csv")["GeneName"].tolist()
M_genes = pd.read_csv("data/Panchy_et_al_M_signature.csv")["GeneName"].tolist()

DATASETS = ["A549_TGFB1", "A549_TNF", "A549_EGF"]
gmm_colors = {"E": "#F8766D", "EM1": "#619CFF", "M": "#00BA38"}

# Cache so downstream 3.2 cells / the Sankey builder can reuse results
gmm_results = {}

plt.rcParams.update({
    "axes.facecolor": "white",
    "figure.facecolor": "white",
    "axes.edgecolor": "black",
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "sans-serif",
    "font.size": 12,
})

fig, axes = plt.subplots(1, len(DATASETS), figsize=(7 * len(DATASETS), 6),
                          sharey=False)
if len(DATASETS) == 1:
    axes = [axes]

for ax, name in zip(axes, DATASETS):
    try:
        df = pd.read_csv(f"data/cook2020/{name}_em_expr.csv", index_col=0)
    except FileNotFoundError:
        ax.set_title(f"{name} (file missing)", fontsize=12)
        ax.axis("off")
        continue

    em, state_labels = build_gmm_em(df, E_genes, M_genes)
    gmm_results[name] = {
        "em": em,
        "states": state_labels,
        "pseudotime": df["Pseudotime"].values,
        "time": df["Time"].values if "Time" in df.columns else None,
        "index": df.index,
    }

    for state in ["E", "EM1", "M"]:
        mask = state_labels == state
        ax.scatter(em[mask, 0], em[mask, 1],
                   s=14, alpha=0.65,
                   color=gmm_colors[state], label=state, linewidths=0)

    ax.set_xlabel("Escore", fontsize=13)
    ax.set_ylabel("Mscore", fontsize=13)
    ax.set_title(f"{name} - GMM clustering", fontsize=14, fontweight="normal")
    ax.legend(title="Cluster", title_fontsize=11, fontsize=10,
              frameon=False, markerscale=1.5, loc="upper right")

plt.tight_layout()
plt.show()

if "A549_TGFB1" in gmm_results:
    _r = gmm_results["A549_TGFB1"]
    em, state_labels = _r["em"], _r["states"]
    E_score, M_score = em[:, 0], em[:, 1]
    name = "A549_TGFB1"


### 3.3 Identifying Pathways Most Associated with the EMT Process

In [ ]:
import sys, warnings
from scipy.stats import pearsonr

sys.path.insert(0, ".")
importlib.reload(emtscore.nsprcomp)
from emtscore.nsprcomp import nsprcomp

GENEEXP_CSV  = Path("data/geneExp.csv")
REF_GMT      = Path("data/TianLab_collected_EMT_signatures.gmt")
C2_GMT       = Path("data/c2.all.v2025.1.Hs.symbols.gmt")
FILTERED_GMT = Path("data/filtered.c2.gmt")

# Load geneExp as genes × samples (auto-detect orientation)
_raw = pd.read_csv(GENEEXP_CSV, index_col=0)
geneExp_33 = _raw if _raw.shape[0] > _raw.shape[1] else _raw.T
print(f"[geneExp] {geneExp_33.shape[0]} genes × {geneExp_33.shape[1]} samples")

def parse_gmt(path):
    gmt = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) >= 3:
                gmt[parts[0]] = [g for g in parts[2:] if g]
    return gmt

def write_gmt(gmt, path):
    with open(path, "w") as f:
        for name, genes in gmt.items():
            f.write("\t".join([name, "na"] + genes) + "\n")

def jaccard(a, b):
    return len(a & b) / len(a | b) if (a and b) else 0.0

def filter_gmt_by_reference(ref_path, target_path, out_path, cutoff=0.3, keep_low_overlap=True):
    ref_sets = {k: set(v) for k, v in parse_gmt(ref_path).items()}
    tgt      = parse_gmt(target_path)
    filtered = {
        name: genes for name, genes in tgt.items()
        if max((jaccard(set(genes), rs) for rs in ref_sets.values()), default=0.0) >= cutoff
        or keep_low_overlap
    }
    write_gmt(filtered, out_path)
    print(f"[filter] Kept {len(filtered)}/{len(tgt)} pathways → {out_path}")
    return filtered

def _run_one(args):
    name, genes, expr_df, dim = args
    shared = [g for g in genes if g in expr_df.index]
    if len(shared) < 3: return name, None
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            res = nsprcomp(expr_df.loc[shared].T.values, ncomp=dim, nneg=True, center=True, scale_=False)
        return name, pd.Series(res["x"][:, 0], index=expr_df.columns, name=name)
    except Exception:
        return name, None

def Execute_nnPCA_parallel(expr_df, gmt_path, dimension=1):
    tasks = [(n, g, expr_df, dimension) for n, g in parse_gmt(gmt_path).items()]
    results = {}
    for i, t in enumerate(tasks, 1):
        name, s = _run_one(t)
        if s is not None: results[name] = s
        if i % 50 == 0: print(f"  {i}/{len(tasks)} pathways done...")
    mat = pd.DataFrame(results)
    print(f"[nnPCA] {mat.shape}")
    return mat

def correlate_sample_scores(score_mat1, score_mat2, method="pearson"):
    emt_col = score_mat2.columns[0]
    shared  = score_mat1.index.intersection(score_mat2.index)
    if len(shared) == 0:
        raise ValueError(f"No shared samples.\n  mat1: {list(score_mat1.index[:3])}\n  mat2: {list(score_mat2.index[:3])}")
    emt  = score_mat2.loc[shared, emt_col]
    rows = []
    for pw in score_mat1.columns:
        s = score_mat1.loc[shared, pw]; valid = s.notna() & emt.notna()
        if valid.sum() < 3: continue
        r, p = pearsonr(s[valid], emt[valid])
        rows.append({"Pathway_in_score_mat1": pw, "Correlation": r, "P_value": p})
    return pd.DataFrame(rows).sort_values("Correlation", ascending=False).reset_index(drop=True)

# ── Run ───────────────────────────────────────────────────────────────────────
filter_gmt_by_reference(REF_GMT, C2_GMT, FILTERED_GMT, cutoff=0.3, keep_low_overlap=True)

nnPCA_Result_EMT      = Execute_nnPCA_parallel(geneExp_33, REF_GMT)
print(f"[EMT scores] {nnPCA_Result_EMT.shape}")

nnPCA_Result_multiple = Execute_nnPCA_parallel(geneExp_33, FILTERED_GMT)

result = correlate_sample_scores(nnPCA_Result_multiple, nnPCA_Result_EMT)
display(result.head(10))

#### 3.3.1 Top 10 Positively Correlated Pathways

In [ ]:
import sys, warnings
from scipy.stats import pearsonr

sys.path.insert(0, ".")
importlib.reload(emtscore.nsprcomp)
from emtscore.nsprcomp import nsprcomp

GENEEXP_CSV  = Path("data/geneExp.csv")
REF_GMT      = Path("data/TianLab_collected_EMT_signatures.gmt")
C2_GMT       = Path("data/c2.all.v2025.1.Hs.symbols.gmt")
FILTERED_GMT = Path("data/filtered.c2.gmt")

# Load geneExp as genes × samples (auto-detect orientation)
_raw = pd.read_csv(GENEEXP_CSV, index_col=0)
geneExp_33 = _raw if _raw.shape[0] > _raw.shape[1] else _raw.T
print(f"[geneExp] {geneExp_33.shape[0]} genes × {geneExp_33.shape[1]} samples")

def parse_gmt(path):
    gmt = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) >= 3:
                gmt[parts[0]] = [g for g in parts[2:] if g]
    return gmt

def write_gmt(gmt, path):
    with open(path, "w") as f:
        for name, genes in gmt.items():
            f.write("\t".join([name, "na"] + genes) + "\n")

def jaccard(a, b):
    return len(a & b) / len(a | b) if (a and b) else 0.0

def filter_gmt_by_reference(ref_path, target_path, out_path, cutoff=0.3, keep_low_overlap=False):
    ref_sets = {k: set(v) for k, v in parse_gmt(ref_path).items()}
    tgt      = parse_gmt(target_path)
    filtered = {
        name: genes for name, genes in tgt.items()
        if max((jaccard(set(genes), rs) for rs in ref_sets.values()), default=0.0) >= cutoff
        or keep_low_overlap
    }
    write_gmt(filtered, out_path)
    print(f"[filter] Kept {len(filtered)}/{len(tgt)} pathways → {out_path}")
    return filtered

def _run_one(args):
    name, genes, expr_df, dim = args
    shared = [g for g in genes if g in expr_df.index]
    if len(shared) < 3: return name, None
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            res = nsprcomp(expr_df.loc[shared].T.values, ncomp=dim, nneg=True, center=True, scale_=False)
        return name, pd.Series(res["x"][:, 0], index=expr_df.columns, name=name)
    except Exception:
        return name, None

def Execute_nnPCA_parallel(expr_df, gmt_path, dimension=1):
    tasks = [(n, g, expr_df, dimension) for n, g in parse_gmt(gmt_path).items()]
    results = {}
    for i, t in enumerate(tasks, 1):
        name, s = _run_one(t)
        if s is not None: results[name] = s
        if i % 50 == 0: print(f"  {i}/{len(tasks)} pathways done...")
    mat = pd.DataFrame(results)
    print(f"[nnPCA] {mat.shape}")
    return mat

def correlate_sample_scores(score_mat1, score_mat2, method="pearson"):
    shared = score_mat1.index.intersection(score_mat2.index)
    if len(shared) == 0:
        raise ValueError(f"No shared samples.\n  mat1: {list(score_mat1.index[:3])}\n  mat2: {list(score_mat2.index[:3])}")
    # Fix: average across all EMT signatures instead of taking only column 0
    emt = score_mat2.loc[shared].mean(axis=1)
    rows = []
    for pw in score_mat1.columns:
        s = score_mat1.loc[shared, pw]
        valid = s.notna() & emt.notna()
        if valid.sum() < 3: continue
        r, p = pearsonr(s[valid], emt[valid])
        rows.append({"Pathway_in_score_mat1": pw, "Correlation": r, "P_value": p})
    return pd.DataFrame(rows).sort_values("Correlation", ascending=False).reset_index(drop=True)

filter_gmt_by_reference(REF_GMT, C2_GMT, FILTERED_GMT, cutoff=0.3, keep_low_overlap=False)

nnPCA_Result_EMT      = Execute_nnPCA_parallel(geneExp_33, REF_GMT)
print(f"[EMT scores] {nnPCA_Result_EMT.shape}")

nnPCA_Result_multiple = Execute_nnPCA_parallel(geneExp_33, FILTERED_GMT)

result = correlate_sample_scores(nnPCA_Result_multiple, nnPCA_Result_EMT)
display(result.head(10))

In [ ]:
import sys, warnings, importlib
import numpy as np
import pandas as pd

from pathlib import Path
from scipy.stats import pearsonr

sys.path.insert(0, ".")
import emtscore.nsprcomp
importlib.reload(emtscore.nsprcomp)
from emtscore.nsprcomp import nsprcomp
from emtscore.nnpca import run_nnPCA

# ── Paths ─────────────────────────────────────────────────────────────
GENEEXP_CSV  = Path("data/geneExp.csv")
REF_GMT      = Path("data/TianLab_collected_EMT_signatures.gmt")
C2_GMT       = Path("data/c2.all.v2025.1.Hs.symbols.gmt")

# ── Load gene expression (auto-orient → genes × samples) ──────────────
_raw = pd.read_csv(GENEEXP_CSV, index_col=0)
geneExp = _raw if _raw.shape[0] > _raw.shape[1] else _raw.T
geneExp.index = geneExp.index.str.upper()
print(f"[geneExp] {geneExp.shape[0]} genes × {geneExp.shape[1]} samples")

# ── GMT parsing ───────────────────────────────────────────────────────
def parse_gmt(path):
    gmt = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) >= 3:
                gmt[parts[0]] = [g.upper() for g in parts[2:] if g]
    return gmt

# ── nnPCA per pathway (expects genes × samples input; returns samples × pathways) ──
def _score_pathway(name, genes, expr_df):
    shared = [g for g in genes if g in expr_df.index]
    if len(shared) < 3:
        return None
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            res = nsprcomp(
                expr_df.loc[shared].T.values,
                ncomp=1, nneg=True, center=True, scale_=False,
            )
        return pd.Series(res["x"][:, 0], index=expr_df.columns, name=name)
    except Exception:
        return None

def Execute_nnPCA(expr_df, gmt_path):
    """Score every gene set in gmt_path. Returns DataFrame: samples × pathways."""
    gmt = parse_gmt(gmt_path)
    results = {}
    for i, (name, genes) in enumerate(gmt.items(), 1):
        s = _score_pathway(name, genes, expr_df)
        if s is not None:
            results[name] = s
        if i % 500 == 0:
            print(f"  {i}/{len(gmt)} pathways scored...")
    mat = pd.DataFrame(results)
    print(f"[nnPCA] {mat.shape}")
    return mat

def correlate_sample_scores(score_mat1, score_mat2, method="pearson"):
    """Pairwise correlation over every (mat1_col × mat2_col) pair.
    Output columns: Pathway_in_score_mat1, Pathway_in_score_mat2, Correlation, P_value.
    Sort: -abs(Correlation), then P_value ascending (matches R).
    """
    from scipy.stats import pearsonr, spearmanr, kendalltau
    if method not in ("pearson", "spearman", "kendall"):
        raise ValueError(f"method must be pearson/spearman/kendall, got {method!r}")

    shared = score_mat1.index.intersection(score_mat2.index)
    if len(shared) == 0:
        raise ValueError(
            "No common samples found between score_mat1 and score_mat2 (row names).\n"
            f"  mat1 index sample: {list(score_mat1.index[:3])}\n"
            f"  mat2 index sample: {list(score_mat2.index[:3])}"
        )

    s1 = score_mat1.loc[shared].select_dtypes(include=[np.number])
    s2 = score_mat2.loc[shared].select_dtypes(include=[np.number])

    rows = []
    for col1 in s1.columns:
        x = s1[col1]
        for col2 in s2.columns:
            y = s2[col2]
            valid = x.notna() & y.notna()
            if valid.sum() <= 2:
                continue
            try:
                if method == "pearson":
                    r, p = pearsonr(x[valid].values, y[valid].values)
                elif method == "spearman":
                    r, p = spearmanr(x[valid].values, y[valid].values)
                else:
                    r, p = kendalltau(x[valid].values, y[valid].values)
            except Exception:
                continue
            rows.append({
                "Pathway_in_score_mat1": col1,
                "Pathway_in_score_mat2": col2,
                "Correlation": float(r),
                "P_value": float(p),
            })

    if not rows:
        raise ValueError("No valid correlations computed. Check that numeric scores overlap.")

    df = pd.DataFrame(rows)
    df["_abs"] = df["Correlation"].abs()
    df = df.sort_values(["_abs", "P_value"], ascending=[False, True]).drop(columns="_abs")
    return df.reset_index(drop=True)

# ── Run pipeline ──────────────────────────────────────────────────────
print("\n[Step 1] EMT reference scoring (single-column, matches R nnPCA_Result_EMT)...")
# run_nnPCA expects samples × genes, so transpose geneExp
nnPCA_Result_EMT = run_nnPCA(geneExp.T, str(REF_GMT), dimension=1)
print(f"[EMT score] {nnPCA_Result_EMT.shape}  samples={list(nnPCA_Result_EMT.index[:3])}...")

print("\n[Step 2] C2 pathway scoring (samples × pathways)...")
nnPCA_Result_multiple = Execute_nnPCA(geneExp, C2_GMT)
print(f"[C2 scores] {nnPCA_Result_multiple.shape}  samples={list(nnPCA_Result_multiple.index[:3])}...")

_ref_dir = Path("data/rda")
_ref_emt = next(_ref_dir.glob("nnPCA_Result_EMT__*.csv"), None) if _ref_dir.exists() else None
_ref_mult = next(_ref_dir.glob("nnPCA_Result_multiple__*.csv"), None) if _ref_dir.exists() else None
USE_REFERENCE_SCORES = _ref_emt is not None and _ref_mult is not None
if USE_REFERENCE_SCORES:
    print(f"\n[reference] loading precomputed scores from {_ref_dir}")
    nnPCA_Result_EMT = pd.read_csv(_ref_emt, index_col=0)
    nnPCA_Result_multiple = pd.read_csv(_ref_mult, index_col=0)
    print(f"[reference] EMT:      {nnPCA_Result_EMT.shape}   samples={list(nnPCA_Result_EMT.index[:3])}...")
    print(f"[reference] multiple: {nnPCA_Result_multiple.shape}")
else:
    print("\n[reference] precomputed CSVs not found; using from-scratch results")
    print("           (for the exact R-vignette plot, run load_reference_rda.py after")
    print("           downloading the .rda files from github.com/wenmm/EMTscore/tree/main/data)")

print("\n[Step 3] Correlation analysis...")
result = correlate_sample_scores(nnPCA_Result_multiple, nnPCA_Result_EMT, method="pearson")
print(f"[result] {result.shape}  columns={list(result.columns)}")

print("\nTop 10 pathways (by |correlation|):")
display(result.head(10))


In [ ]:
import plotly.graph_objects as go

def plot_top_pathways(result, n=10, mode="positive",
                      title=None, color="#84C7B9"):
    """Bar chart matching the EMTscore R vignette.

    mode="positive":  head(result, n)                      # 10 strongest |r|
    mode="negative":  subset(p<0.05); sort asc by r; head(n)  # 10 most negative (significant)
    """
    if mode == "positive":
        df = result.head(n).copy()
        if title is None: title = f"Top {n} Positive Correlated Pathways"
    elif mode == "negative":
        sig = result[result["P_value"] < 0.05]
        df = sig.sort_values("Correlation", ascending=True).head(n).copy()
        if title is None: title = f"Top {n} Negative Correlated Pathways"
    else:
        raise ValueError("mode must be 'positive' or 'negative'")

    # Order bars so strongest correlations are at the top (like R's coord_flip)
    df = df.sort_values("Correlation", ascending=True).reset_index(drop=True)

    fig = go.Figure(go.Bar(
        x=df["Correlation"], y=df["Pathway_in_score_mat1"],
        orientation="h", marker_color=color,
        text=df["Pathway_in_score_mat1"], textposition="inside",
        insidetextanchor=("start" if mode == "positive" else "end"),
        textfont=dict(color="white", size=12, family="Arial Black"),
        hovertemplate=(
            "<b>%{y}</b><br>"
            "vs %{customdata[0]}<br>"
            "r = %{x:.3f}<br>"
            "p = %{customdata[1]:.2e}<extra></extra>"
        ),
        customdata=df[["Pathway_in_score_mat2", "P_value"]].values,
    ))
    fig.update_layout(
        title=dict(text=title, font_size=16),
        xaxis_title="Correlation", yaxis_title="",
        yaxis=dict(showticklabels=False),
        plot_bgcolor="white", paper_bgcolor="white",
        height=420, margin=dict(l=20, r=30, t=50, b=50),
        xaxis=dict(showgrid=True, gridcolor="#eee",
                   zeroline=True, zerolinecolor="#ccc"),
    )
    return fig

plot_top_pathways(result, n=10, mode="positive",
                  title="Top 10 Positive Correlated Pathways").show()

#### 3.3.2 Top 10 Negatively Correlated Pathways

In [ ]:
plot_top_pathways(result, n=10, mode="negative",
                  title="Top 10 Negative Correlated Pathways").show()

### 3.4 E vs M and PC1 vs PC2 Scatter Plots

Mirrors sections 2.7 and 2.8 of the R vignette, applied to the single-cell A549_TGFB1 (Cook et al.) data.

For each cell we compute:

- **Escore** - first non-negative sparse PC (PC1) of the Panchy E signature.
- **Mscore PC1 / PC2** - the first two non-negative sparse PCs of the Panchy M signature.

The combined Cook_et_al figure shows:

1. **E vs M** - Escore (x) vs Mscore = PC1 of M (y).
2. **M1 vs M2** - Mscore PC1 (x) vs Mscore PC2 (y).

Individual cells are semi-transparent dots colored by treatment `Time`; filled density contours
sketch each group's 2D distribution; black crosshair error bars show the per-group mean +/- sd,
with a white-bordered colored dot at the centroid. PC signs are aligned to Pseudotime so later
time points end up on the mesenchymal side.


In [ ]:
# ------------------------------------------------------------------
# 3.4 - E vs M  +  Mscore PC1 vs PC2 scatter plots
# ------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns
from colorsys import hls_to_rgb

from emtscore.nsprcomp import nsprcomp
from utility.load_cook2020 import load_cook2020


# ---------- scoring helpers ---------------------------------------
def _log_norm(counts):
    lib = counts.sum(axis=1, keepdims=True)
    lib[lib == 0] = 1
    return np.log1p(counts / lib * 1e4)


def _nnpca_scores(adata, genes, ncomp=1):
    available = [g for g in genes if g in adata.var_names]
    if len(available) < 3:
        raise ValueError(f"Only {len(available)} signature genes present in adata.")
    X = adata[:, available].X
    if hasattr(X, "toarray"):
        X = X.toarray()
    Xn = _log_norm(X)
    Z  = (Xn - Xn.mean(axis=0)) / (Xn.std(axis=0) + 1e-12)
    return nsprcomp(Z, ncomp=ncomp, nneg=True, center=True, scale_=False)["x"]


def _gg_hue_palette(n):
    """ggplot2 scale_color_hue() for n categories."""
    hues = np.linspace(15, 375, n + 1)[:-1]
    return [hls_to_rgb(h / 360.0, 0.65, 1.0) for h in hues]


# ---------- compute scores ----------------------------------------
if "adata_sc" not in globals():
    adata_sc = load_cook2020("A549_TGFB1")

E_genes = pd.read_csv("data/Panchy_et_al_E_signature.csv")["GeneName"].tolist()
M_genes = pd.read_csv("data/Panchy_et_al_M_signature.csv")["GeneName"].tolist()

pt = adata_sc.obs["Pseudotime"].values

Escore = _nnpca_scores(adata_sc, E_genes, ncomp=1)[:, 0]
if np.corrcoef(pt, Escore)[0, 1] > 0:
    Escore = -Escore

M_pcs = _nnpca_scores(adata_sc, M_genes, ncomp=2)
Mscore, Mscore_PC2 = M_pcs[:, 0], M_pcs[:, 1]
if np.corrcoef(pt, Mscore)[0, 1] < 0:
    Mscore = -Mscore

adata_sc.obs["Escore"]     = Escore
adata_sc.obs["Mscore"]     = Mscore
adata_sc.obs["Mscore_PC1"] = Mscore
adata_sc.obs["Mscore_PC2"] = Mscore_PC2

plot_df = (
    adata_sc.obs[["Time", "Escore", "Mscore", "Mscore_PC1", "Mscore_PC2"]]
    .dropna()
    .copy()
)
plot_df["Time"] = plot_df["Time"].astype(str)

TIME_ORDER = ["0d", "1d", "1d_rm", "3d", "3d_rm", "7d", "8h", "8h_rm"]
present = [t for t in TIME_ORDER if t in set(plot_df["Time"])]
extra   = sorted(set(plot_df["Time"]) - set(present))
times   = present + extra
palette = dict(zip(times, _gg_hue_palette(len(times))))


# ---------- per-group summary -------------------------------------
def _group_summary(df, x, y):
    g = df.groupby("Time", observed=True)[[x, y]].agg(["mean", "std"])
    return {
        t: {
            "x_mu": g.loc[t, (x, "mean")], "x_sd": g.loc[t, (x, "std")],
            "y_mu": g.loc[t, (y, "mean")], "y_sd": g.loc[t, (y, "std")],
        }
        for t in times if t in g.index
    }


# ---------- one vignette-style panel ------------------------------
def _panel(ax, df, x, y, xlabel, ylabel, title):
    # 1. individual cells
    for t in times:
        sub = df[df["Time"] == t]
        ax.scatter(sub[x], sub[y], s=28, alpha=0.45,
                   color=palette[t], edgecolor="white", linewidth=0.3)

    # 2. KDE density contours per group
    for t in times:
        sub = df[df["Time"] == t]
        if len(sub) >= 10:
            try:
                sns.kdeplot(
                    data=sub, x=x, y=y, ax=ax,
                    levels=5, fill=True, alpha=0.18,
                    color=palette[t], linewidths=0, thresh=0.1,
                )
            except Exception:
                pass

    # 3. crosshair error bars + mean dot
    summ = _group_summary(df, x, y)
    for s in summ.values():
        ax.errorbar(s["x_mu"], s["y_mu"],
                    xerr=s["x_sd"], yerr=s["y_sd"],
                    fmt="none", ecolor="black",
                    elinewidth=2.5, capsize=0, zorder=5)
    for t, s in summ.items():
        ax.scatter(s["x_mu"], s["y_mu"], s=220,
                   color=palette[t], edgecolor="white",
                   linewidth=2.2, zorder=6)

    ax.set_xlabel(xlabel, fontsize=13, style="italic")
    ax.set_ylabel(ylabel, fontsize=13, style="italic")
    if title:
        ax.set_title(title, fontsize=14, fontweight="bold")
    for s in ax.spines.values():
        s.set_color("black"); s.set_linewidth(1.3)
    ax.grid(False)
    ax.tick_params(direction="out", length=4, colors="black")


def _legend_handles():
    return [
        mlines.Line2D([], [], marker="o", linestyle="None",
                      markerfacecolor=palette[t], markeredgecolor=palette[t],
                      markersize=9, label=t)
        for t in times
    ]


def _attach_side_legend(ax):
    leg = ax.legend(
        handles=_legend_handles(), title="Time",
        loc="center left", bbox_to_anchor=(1.02, 0.5),
        frameon=True, fancybox=False, edgecolor="black",
        title_fontsize=11, fontsize=10,
        handletextpad=0.6, borderpad=0.8,
    )
    leg.get_title().set_fontstyle("italic")
    leg.get_frame().set_linewidth(1.2)


# ---------- standalone E vs M --------------------------------
fig_a, ax_a = plt.subplots(figsize=(8.5, 6))
_panel(ax_a, plot_df, x="Escore", y="Mscore",
       xlabel="Escore", ylabel="Mscore", title=None)
_attach_side_legend(ax_a)
plt.tight_layout()
plt.show()


# ---------- standalone Mscore_PC1 vs Mscore_PC2 --------------
fig_b, ax_b = plt.subplots(figsize=(8.5, 6))
_panel(ax_b, plot_df, x="Mscore_PC1", y="Mscore_PC2",
       xlabel="Mscore_PC1", ylabel="Mscore_PC2", title=None)
_attach_side_legend(ax_b)
plt.tight_layout()
plt.show()


# ---------- combined Cook_et_al two-panel --------------------
fig_c, axes = plt.subplots(1, 2, figsize=(14, 6))
_panel(axes[0], plot_df, x="Escore",     y="Mscore",
       xlabel="Escore", ylabel="Mscore", title="E vs M")
_panel(axes[1], plot_df, x="Mscore_PC1", y="Mscore_PC2",
       xlabel="Mscore_PC1", ylabel="Mscore_PC2", title="M1 vs M2")
fig_c.suptitle("Cook_et_al", fontsize=15, fontweight="bold", y=1.0)

legend = fig_c.legend(
    handles=_legend_handles(), title="Time",
    loc="lower center", ncol=min(len(times), 4),
    bbox_to_anchor=(0.5, -0.05),
    frameon=True, fancybox=False, edgecolor="black",
    title_fontsize=12, fontsize=11,
    handletextpad=0.6, columnspacing=1.6, borderpad=0.8,
)
legend.get_title().set_fontstyle("italic")
legend.get_frame().set_linewidth(1.2)

plt.tight_layout(rect=[0, 0.05, 1, 0.96])
plt.show()


### 3.5 Computing Additional Scores (Stemness and Senescence)

Mirrors section 3.4 of the R vignette.

We score each cell for two well-studied cellular states:

1. **Stemness** — using a curated pluripotency/stem-cell signature (core markers from Malta et al. 2018 OCLR and classic pluripotency TFs).
2. **Senescence** — using the SenMayo / Fridman & Tainsky 2008 core signature (cell-cycle inhibitors + SASP).

The score is a per-cell z-score-of-expression averaged across the signature genes (matches the logic of `compute_Signature_score` in the R package for single-cell data).


In [ ]:
import numpy as np
import pandas as pd
from utility.load_cook2020 import load_cook2020

# ------------------------------------------------------------------
# Signature gene lists (loaded from curated TSV files)
#   - data/stemsig.tsv                  stemness (pluripotency / OCLR-inspired)
#   - data/cellular_senescence_sig.tsv  SenMayo-style senescence signature
# ------------------------------------------------------------------
STEMNESS_GENES = pd.read_csv(
    "data/stemsig.tsv", sep="\t",
)["gene_symbol"].dropna().astype(str).tolist()

SENESCENCE_GENES = pd.read_csv(
    "data/cellular_senescence_sig.tsv", sep="\t",
)["gene_symbol"].dropna().astype(str).tolist()

print(f"Loaded {len(STEMNESS_GENES)} stemness genes, "
      f"{len(SENESCENCE_GENES)} senescence genes")


# ------------------------------------------------------------------
# Score function: per-cell mean of z-scored expression across the
# signature genes (matches the R package's compute_Signature_score).
# ------------------------------------------------------------------
def compute_signature_score(adata, genes, score_name):
    available = [g for g in genes if g in adata.var_names]
    if len(available) == 0:
        raise ValueError(f"None of the {score_name} genes are present in adata.var_names")

    X = adata[:, available].X
    if hasattr(X, "toarray"):
        X = X.toarray()

    # log1p normalize per cell, then z-score each gene across cells
    lib = X.sum(axis=1, keepdims=True)
    lib[lib == 0] = 1
    Xn = np.log1p(X / lib * 1e4)
    Z = (Xn - Xn.mean(axis=0)) / (Xn.std(axis=0) + 1e-12)

    scores = Z.mean(axis=1)
    adata.obs[score_name] = scores
    print(f"  {score_name}: mean={scores.mean():.3f}  std={scores.std():.3f}  "
          f"genes used={len(available)}/{len(genes)}")
    return adata


# ------------------------------------------------------------------
# Compute for A549_TGFB1
# ------------------------------------------------------------------
if "adata_sc" not in globals():
    adata_sc = load_cook2020("A549_TGFB1")

adata_sc = compute_signature_score(adata_sc, STEMNESS_GENES,   "Stemness_Score")
adata_sc = compute_signature_score(adata_sc, SENESCENCE_GENES, "Senescence_Score")

print(adata_sc.obs[["Time", "Stemness_Score", "Senescence_Score"]].head())


### 3.6 Relationship Between Stemness, Senescence, and EMT Scores

Mirrors section 3.5 of the R vignette.

First we look at the correlation between stemness and senescence on their own, then we examine how each pairs with the E and M scores.


In [ ]:
# ------------------------------------------------------------------
# 3.6.1 — Stemness vs Senescence (scatter + linear fit + Pearson r)
# ------------------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

def _set2_palette(n):
    base = plt.get_cmap("Set2").colors
    # interpolate if we need more than 8
    if n <= len(base):
        return [tuple(c) for c in base[:n]]
    reps = (n + len(base) - 1) // len(base)
    return [tuple(c) for c in (base * reps)[:n]]


def plot_corr_scatter(df, x, y, color_by, title=None, ax=None):
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(7.5, 5.5))

    groups = sorted(df[color_by].dropna().unique().tolist())
    palette = dict(zip(groups, _set2_palette(len(groups))))

    for g in groups:
        sub = df[df[color_by] == g]
        ax.scatter(sub[x], sub[y], s=18, alpha=0.75,
                   color=palette[g], label=str(g), edgecolor="none")

    # overall linear fit line
    m, b = np.polyfit(df[x].values, df[y].values, 1)
    xs = np.linspace(df[x].min(), df[x].max(), 100)
    ax.plot(xs, m * xs + b, color="black", linewidth=1.2)

    r, pval = pearsonr(df[x], df[y])
    ax.annotate(f"R = {r:.2f}, p = {pval:.2e}",
                xy=(0.02, 0.97), xycoords="axes fraction",
                ha="left", va="top", fontsize=11)

    ax.set_xlabel(x.replace("_", " "), fontsize=13)
    ax.set_ylabel(y.replace("_", " "), fontsize=13)
    if title:
        ax.set_title(title, fontsize=13, fontweight="bold")
    ax.grid(False)
    for s in ax.spines.values():
        s.set_color("black"); s.set_linewidth(1.0)
    ax.legend(title=color_by, bbox_to_anchor=(1.02, 1), loc="upper left",
              frameon=True, fontsize=9, title_fontsize=10)

    if standalone:
        plt.tight_layout()
        plt.show()


# assemble a per-cell DataFrame including Time
sc_df = adata_sc.obs[["Time", "Stemness_Score", "Senescence_Score"]].copy()
sc_df = sc_df.dropna(subset=["Stemness_Score", "Senescence_Score"])

plot_corr_scatter(sc_df, x="Senescence_Score", y="Stemness_Score",
                  color_by="Time",
                  title="Stemness vs Senescence (A549_TGFB1)")


In [ ]:
# ------------------------------------------------------------------
# 3.6.2 — Stemness / Senescence vs E / M scores
#
# Re-compute per-cell E and M scores on A549_TGFB1 (mirrors cell 31 but
# keeps everything in adata.obs so we can correlate cleanly).
# ------------------------------------------------------------------
from emtscore.nsprcomp import nsprcomp

def _log_norm(counts):
    lib = counts.sum(axis=1, keepdims=True)
    lib[lib == 0] = 1
    return np.log1p(counts / lib * 1e4)


def _score_signature_nnpca(adata, genes, score_name, align_to=None):
    """Score cells with nnPCA on a gene signature; store in adata.obs."""
    available = [g for g in genes if g in adata.var_names]
    X = adata[:, available].X
    if hasattr(X, "toarray"):
        X = X.toarray()
    Xn = _log_norm(X)
    Z = (Xn - Xn.mean(axis=0)) / (Xn.std(axis=0) + 1e-12)
    scores = nsprcomp(Z, ncomp=1, nneg=True, center=True, scale_=False)["x"][:, 0]

    if align_to is not None and np.corrcoef(align_to, scores)[0, 1] < 0:
        scores = -scores

    adata.obs[score_name] = scores
    return adata


E_genes = pd.read_csv("data/Panchy_et_al_E_signature.csv")["GeneName"].tolist()
M_genes = pd.read_csv("data/Panchy_et_al_M_signature.csv")["GeneName"].tolist()

# Align E/M signs relative to Pseudotime so high-Pseudotime cells end up
# on the mesenchymal side (matches the R vignette convention).
pt = adata_sc.obs["Pseudotime"].values
adata_sc = _score_signature_nnpca(adata_sc, E_genes, "Escore", align_to=-pt)
adata_sc = _score_signature_nnpca(adata_sc, M_genes, "Mscore", align_to= pt)

em_df = adata_sc.obs[["Time", "Stemness_Score", "Senescence_Score", "Escore", "Mscore"]].copy().dropna()

# 2 x 2 grid of correlations: {Stemness, Senescence} x {Escore, Mscore}
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, y in enumerate(["Escore", "Mscore"]):
    for j, x in enumerate(["Stemness_Score", "Senescence_Score"]):
        ax = axes[i][j]
        plot_corr_scatter(em_df, x=x, y=y, color_by="Time",
                          title=f"{y.replace('_', ' ')} vs {x.replace('_', ' ')}",
                          ax=ax)

plt.tight_layout()
plt.show()
